# 06 · Final pipeline: summary of all models, inference strategies, ensembling, submission
1. Summary of every model (same protocol). 2. **Inference / data strategy** on the best anchor-free model
(no retraining): test-time augmentation and tiled (sliced) inference. 3. **Model strategy**: Weighted Boxes
Fusion of the tuned models of each family (CPU). 4. Both combined. 5. The two best candidates by val mAP become the
two final submissions. Needs the tuned runs of 02-05 (their val + test predictions in WORK_DIR/preds).

In [1]:
# Paths: auto-detected for Colab, Kaggle and a local clone. Override by hand if your layout differs.
import os, sys
from pathlib import Path
if "google.colab" in sys.modules:
    ENV = "colab"
    from google.colab import drive
    drive.mount("/content/drive")                                   # outputs survive disconnects
    DATA_ROOT = Path("/content/data")                               # folder with test_images.csv
    REPO_DIR, WORK_DIR = Path("/content/Video-Detection-HW2"), Path("/content/drive/MyDrive/hw2_outputs")
elif os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):                      # Colab also has /kaggle/input -> check env var
    ENV = "kaggle"
    DATA_ROOT = next(Path("/kaggle/input").glob("**/test_images.csv")).parent  # competition data
    REPO_DIR, WORK_DIR = Path("/kaggle/working/Video-Detection-HW2"), Path("/kaggle/working/outputs")
else:
    ENV = "local"
    REPO_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    DATA_ROOT, WORK_DIR = REPO_DIR / "data", REPO_DIR / "outputs"
assert (DATA_ROOT / "test_images.csv").exists(), f"test_images.csv not found in {DATA_ROOT}"
print("env:", ENV, "| DATA_ROOT:", DATA_ROOT, "| REPO_DIR:", REPO_DIR, "| WORK_DIR:", WORK_DIR)

env: local | DATA_ROOT: /Users/anastasiiayablunovska/year3/ai/Video-Detection-HW2/data | REPO_DIR: /Users/anastasiiayablunovska/year3/ai/Video-Detection-HW2 | WORK_DIR: /Users/anastasiiayablunovska/year3/ai/Video-Detection-HW2/outputs


In [2]:
# Get the code + pinned dependencies on Kaggle / Colab (needs Internet). Locally nothing happens.
import subprocess
REPO_URL = "https://github.com/ystacy-ab/Video-Detection-HW2.git"
if ENV != "local":
    if (REPO_DIR / ".git").exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=False)
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)

In [3]:
# Make `src/` importable and print library versions (reproducibility)
sys.path.insert(0, str(Path(REPO_DIR).resolve()))
from src.utils import load_config, load_exp_config, seed_everything, print_versions, ensure_dir
print_versions()

python          3.12.6
numpy           2.4.6
torch           2.11.0
torchvision     0.26.0
ultralytics     8.4.208
faster_coco_eval 1.8.0
pycocotools     installed
ensemble_boxes  installed
sahi            0.14.0
cv2             5.0.0
wandb           0.31.0
device          CPU only


In [4]:
# Shared config + this notebook's experiment configs, seeds, output folders
cfg = load_config(Path(REPO_DIR) / "configs" / "base.yaml")
seed_everything(cfg["seed"])
DATA_ROOT, WORK_DIR = Path(DATA_ROOT), ensure_dir(WORK_DIR)
RES_DIR, FIG_DIR = Path(REPO_DIR) / "results", ensure_dir(Path(REPO_DIR) / "results" / "figures")
GT_VAL = DATA_ROOT / "val" / "annotations_coco.json"
EXPS = {n: load_exp_config(Path(REPO_DIR) / "configs" / f"{n}.yaml") for n in ['frcnn_tuned', 'retinanet_tuned', 'yolo_p2', 'rtdetr_tuned']}
RUN = list(EXPS)  # experiments this session may TRAIN; remove names to split the work over sessions
print("experiments:", RUN)

experiments: ['frcnn_tuned', 'retinanet_tuned', 'yolo_p2', 'rtdetr_tuned']


In [5]:
# Imports, GPU check and small helpers used below
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from src.utils import CLASS_NAMES, load_predictions
from src.eval import evaluate, per_class_table, evaluate_groups, clip_groups, density_groups
from src.viz import show_grid
from src import analysis as A
pd.set_option("display.width", 180)
RUN_GPU = torch.cuda.is_available()  # training / GPU-only cells are skipped on CPU
print("GPU:", torch.cuda.get_device_name(0) if RUN_GPU else "none -> training cells will be skipped")


def have(name, split="val"):
    return (WORK_DIR / "preds" / f"{name}_{split}.pkl").exists()


def compare(names, title=None):
    """COCO metrics of saved val predictions (same evaluator for every model)."""
    rows = {n: evaluate(load_predictions(WORK_DIR / "preds" / f"{n}_val.pkl"), GT_VAL, verbose=False)
            for n in names if have(n)}
    cols = ["mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "AR"] + [f"AP_{c}" for c in CLASS_NAMES]
    t = pd.DataFrame(rows, index=cols).T.round(4)
    if title and len(t):
        t.to_csv(RES_DIR / f"{title}.csv")
    return t


def speed_table(names):
    """Speed / training time from results/experiments.csv (last row per experiment)."""
    e = pd.read_csv(RES_DIR / "experiments.csv")
    e = e[e["experiment"].isin(names)].drop_duplicates("experiment", keep="last")
    return e[["experiment", "val_mAP", "AP50", "AP_small", "ms_per_img", "FPS", "train_time_min"]]
from src.ensemble import wbf, image_sizes
from src.models_ul import load_ul, predict_ul, predict_tiled, make_predict_fn_ul, test_lists, write_submission
from src.train_tv import prepare_indices
from src.eval import measure_speed
from src.utils import save_predictions, log_experiment
_, _, val_full_idx = prepare_indices(cfg, DATA_ROOT)
SIZES = image_sizes(val_full_idx, DATA_ROOT / "test_images.csv")
CANDS = {}  # name -> {"type", "val_mAP", "AP_small", "ms_per_img", "test": preds or None}


def add_candidate(name, kind, val_preds, test_preds, ms):
    m = evaluate(val_preds, GT_VAL, verbose=False)
    CANDS[name] = {"type": kind, "val_mAP": m["mAP"], "AP50": m["AP50"], "AP_small": m["AP_small"],
                   "ms_per_img": ms, "test": test_preds, "metrics": m}
    print(f"{name:40s} mAP={m['mAP']:.4f} AP50={m['AP50']:.4f} APs={m['AP_small']:.4f}")

GPU: none -> training cells will be skipped
[index] train: 56 clips, 4062 frames (step=6)
[index] val: 7 clips, 2846 frames (step=1)
[data] train 4062 | val per-epoch 714 | val final 2846


## 1. Summary of all models (same split, same metric)

In [6]:
# Every experiment logged in results/experiments.csv (last row per experiment)
allx = pd.read_csv(RES_DIR / "experiments.csv").drop_duplicates("experiment", keep="last")
allx[["experiment", "model", "val_mAP", "AP50", "AP_small", "AP_medium", "AP_large", "ms_per_img", "FPS", "train_time_min"]]

,experiment,model,val_mAP,AP50,AP_small,AP_medium,AP_large,ms_per_img,FPS,train_time_min


In [7]:
# Single models available for the final pipeline (tuned run of each family with val AND test predictions)
SPEED = dict(zip(allx["experiment"], allx["ms_per_img"]))
for n in EXPS:
    if have(n) and have(n, "test"):
        add_candidate(n, "single model", load_predictions(WORK_DIR / "preds" / f"{n}_val.pkl"),
                      load_predictions(WORK_DIR / "preds" / f"{n}_test.pkl"), SPEED.get(n, np.nan))
    else:
        print(f"{n}: missing val or test predictions -> run its notebook first")

frcnn_tuned: missing val or test predictions -> run its notebook first
retinanet_tuned: missing val or test predictions -> run its notebook first
yolo_p2: missing val or test predictions -> run its notebook first
rtdetr_tuned: missing val or test predictions -> run its notebook first


## 2. Inference strategy (no retraining): TTA and tiled inference on the anchor-free model (GPU)
TTA = Ultralytics flip + 3 scales. Tiles = full frame + 2×2 overlapping tiles (25 %), each upsampled to the
training size, merged by class-wise NMS (simplified SAHI). Both change only the input at test time.

In [8]:
# Run TTA and tiles for yolo_p2 on all val + test frames (cached in WORK_DIR/preds, ~10-20 min each on T4)
UL_NAME = "yolo_p2"
ucfg = EXPS[UL_NAME]
w = WORK_DIR / "ul" / UL_NAME / "weights" / "last.pt"
if w.exists():
    model = load_ul(w, ucfg["model"]["arch"])
    imgsz, pc = ucfg["train"]["imgsz"], ucfg["predict"]
    vk, vp = val_full_idx["key"].tolist(), val_full_idx["img_path"].tolist()
    tk, tp = test_lists(DATA_ROOT)
    for kind, fn in [("tta", lambda k, p, d: predict_ul(model, k, p, pc, imgsz, augment=True, desc=d)),
                     ("tiles", lambda k, p, d: predict_tiled(model, k, p, pc, imgsz, desc=d))]:
        name = f"{UL_NAME}_{kind}"
        if not (have(name) and have(name, "test")):
            if not RUN_GPU:
                print(f"{name}: skipped (no GPU)"); continue
            save_predictions(fn(vk, vp, f"{kind} val"), WORK_DIR / "preds" / f"{name}_val.pkl")
            save_predictions(fn(tk, tp, f"{kind} test"), WORK_DIR / "preds" / f"{name}_test.pkl")
        ms = measure_speed(lambda p: fn([p], [p], kind), vp[::60], warmup=3, iters=20)["ms_per_img"] if RUN_GPU else np.nan
        add_candidate(name, f"inference: {kind}", load_predictions(WORK_DIR / "preds" / f"{name}_val.pkl"),
                      load_predictions(WORK_DIR / "preds" / f"{name}_test.pkl"), ms)
    del model
else:
    print("yolo_p2 weights not found -> run 04_yolo.ipynb first")

yolo_p2 weights not found -> run 04_yolo.ipynb first


## 3. Model strategy: Weighted Boxes Fusion of the families' tuned models (CPU)

In [9]:
# WBF of the 2, 3, 4 best single models (ranked by val mAP); equal weights, IoU 0.6
singles = sorted([n for n, c in CANDS.items() if c["type"] == "single model"], key=lambda n: -CANDS[n]["val_mAP"])
for k in range(2, len(singles) + 1):
    parts = singles[:k]
    v, ms = wbf([load_predictions(WORK_DIR / "preds" / f"{n}_val.pkl") for n in parts], SIZES, desc=f"WBF val {k}")
    t, _ = wbf([CANDS[n]["test"] for n in parts], SIZES, desc=f"WBF test {k}")
    add_candidate("wbf:" + "+".join(parts), "model: WBF", v, t, sum(CANDS[n]["ms_per_img"] for n in parts) + ms)
if len(singles) < 2:
    print("need at least 2 single models for WBF")

need at least 2 single models for WBF


## 4. Both: WBF with the best inference variant replacing the plain anchor-free model

In [10]:
# Best inference variant (TTA or tiles) + the other families' models
inf = sorted([n for n, c in CANDS.items() if c["type"].startswith("inference")], key=lambda n: -CANDS[n]["val_mAP"])
others = [n for n in singles if n != "yolo_p2"]
if inf and others:
    parts = [inf[0]] + others[:2]
    v, ms = wbf([load_predictions(WORK_DIR / "preds" / f"{n}_val.pkl") for n in parts], SIZES, desc="WBF val")
    t, _ = wbf([CANDS[n]["test"] for n in parts], SIZES, desc="WBF test")
    add_candidate("wbf:" + "+".join(parts), "inference + model", v, t, sum(CANDS[n]["ms_per_img"] for n in parts) + ms)
else:
    print("needs an inference variant and at least one other model")

needs an inference variant and at least one other model


## 5. Candidates, final choice (by val mAP) and the two submissions

In [11]:
# All candidates on the same val protocol; the two best become final_1 / final_2 submissions
table = pd.DataFrame([{"candidate": n, **{k: v for k, v in c.items() if k not in ("test", "metrics")}} for n, c in CANDS.items()])
if len(table):
    table = table.sort_values("val_mAP", ascending=False).reset_index(drop=True)
    table.to_csv(RES_DIR / "final_candidates.csv", index=False)
    display(table.round(4))
else:
    print("no candidates yet")

no candidates yet


In [12]:
# Write + validate the two final submissions and log them to results/experiments.csv
for rank, row in enumerate(table.head(2).itertuples(), start=1) if len(table) else []:
    c = CANDS[row.candidate]
    out = write_submission(c["test"], cfg, DATA_ROOT, WORK_DIR, f"final_{rank}")
    log_experiment(f"final_{rank}", c["type"], {"candidate": row.candidate}, c["metrics"],
                   {"ms_per_img": c["ms_per_img"], "fps": 1000 / c["ms_per_img"] if c["ms_per_img"] == c["ms_per_img"] else None},
                   notes=row.candidate)
    print(f"final_{rank}: {row.candidate} -> {out}")